# Homework 6 — Tree-Based Models

## Homework Objective

The objective of this homework is to build and compare **tree-based regression models** for predicting car fuel efficiency.

The notebook covers decision tree splitting, random forest tuning, feature importance analysis, and XGBoost hyperparameter comparison.

**Target:** `fuel_efficiency_mpg`  
**Features:** All available features in the car fuel efficiency dataset  
**Key Models:** Decision Tree, Random Forest, XGBoost

# Cohort Homework-06: Tree-Based Models

## Importing modules

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import mean_squared_error
import xgboost as xgb

## Loading and Preparing Data

In [2]:
df = pd.read_csv(r'D:\ml-zoomcamp-2026\data\car_fuel_efficiency_2026.csv')
df.fillna(0, inplace=True)

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values

df_train = df_train.drop(columns=['fuel_efficiency_mpg'])
df_val = df_val.drop(columns=['fuel_efficiency_mpg'])

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))

In [3]:
dv.get_feature_names_out()

array(['acceleration', 'drivetrain=All-wheel drive',
       'drivetrain=Front-wheel drive', 'drivetrain=Rear-wheel drive',
       'engine_displacement', 'fuel_type=Diesel', 'fuel_type=Gasoline',
       'fuel_type=Hybrid', 'horsepower', 'model_year', 'num_cylinders',
       'num_doors', 'origin=Asia', 'origin=Europe', 'origin=USA',
       'vehicle_weight'], dtype=object)

## Helper Function

In [4]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# Question 1 — Decision Tree Split Feature

A decision tree with `max_depth=1` makes exactly one split on the most informative feature.  
We can inspect `tree_.feature[0]` to find which feature index was chosen, then map it back via `DictVectorizer`.  
If the feature is a one-hot encoded column (e.g. `origin=USA`), we report its original column name (`origin`).

**Which original feature does the tree split on at depth 1?**

In [5]:
# Q1
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

feature_names = dv.get_feature_names_out()
split_feature_index = dt.tree_.feature[0]
split_feature_name = feature_names[split_feature_index]

print(f"Split feature: {split_feature_name}")

Split feature: model_year


### Answer Q1
ans: **`model_year`**

# Question 2 — Random Forest RMSE

Train a `RandomForestRegressor` with `n_estimators=10` and `random_state=1`.  
Evaluate using **RMSE** on the validation set.

**What is the RMSE on the validation dataset?**

In [6]:
# Q2
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_val = rf.predict(X_val)
print(f"RMSE: {rmse(y_val, y_pred_val):.4f}")

RMSE: 1.8332


### Answer Q2
ans: **`1.837`** (closest option)

# Question 3 — Tuning n_estimators

Experiment with `n_estimators` values `[10, 50, 100, 150]` while keeping `random_state=1`.  
Evaluate RMSE on the validation set for each value (compare at 3 decimal places).

**Which value of `n_estimators` gives the lowest validation RMSE?**

In [7]:
# Q3
for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    score = rmse(y_val, rf.predict(X_val))
    print(f"n_estimators={n:3d} → RMSE={score:.3f}")

n_estimators= 10 → RMSE=1.833
n_estimators= 50 → RMSE=1.774
n_estimators=100 → RMSE=1.770
n_estimators=150 → RMSE=1.769


### Answer Q3
ans: **`150`** — lowest RMSE at 3 decimal places (1.769)

# Question 4 — Best max_depth via Mean RMSE

Select the best `max_depth` from `[10, 15, 20, 25]`.  
For each depth, try `n_estimators` in `[10, 50, 100, 150]` and compute the **mean RMSE** across all 4 forest sizes.  
Use `random_state=1` throughout.

**Which `max_depth` gives the lowest mean RMSE?**

In [8]:
# Q4
for depth in [10, 15, 20, 25]:
    scores = []
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(n_estimators=n, max_depth=depth, random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
        scores.append(rmse(y_val, rf.predict(X_val)))
    print(f"max_depth={depth:2d} → mean RMSE={np.mean(scores):.3f}")

max_depth=10 → mean RMSE=1.754
max_depth=15 → mean RMSE=1.784
max_depth=20 → mean RMSE=1.787
max_depth=25 → mean RMSE=1.786


### Answer Q4
ans: **`10`** — lowest mean RMSE (1.754)

# Question 5 — Most Important Feature

Tree-based models expose `feature_importances_`, which reflects how much each feature reduces impurity (gain) across all splits.  

Train a Random Forest with `n_estimators=10`, `max_depth=20`, `random_state=1`.  
Check the importance scores for these four features:
- `vehicle_weight`, `horsepower`, `acceleration`, `engine_displacement`

**Which feature has the highest importance score?**

In [9]:
# Q5
rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

feature_names = dv.get_feature_names_out()
importances = rf.feature_importances_

targets = ['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']
for name in targets:
    idx = list(feature_names).index(name)
    print(f"{name}: {importances[idx]:.4f}")

vehicle_weight: 0.2033
horsepower: 0.0783
acceleration: 0.0550
engine_displacement: 0.0613


### Answer Q5
ans: **`vehicle_weight`** — highest importance (0.2033)

# Question 6 — XGBoost eta Comparison

Train an XGBoost regressor for 100 rounds using the parameters below.  
Compare two values of the learning rate (`eta`): `0.3` and `0.1`.  
Use a watchlist to monitor validation RMSE.

**Which `eta` value leads to the better (lower) validation RMSE?**

In [10]:
# Q6
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)
watchlist = [(dtrain, 'train'), (dval, 'val')]

for eta in [0.3, 0.1]:
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,
        'objective': 'reg:squarederror',
        'nthread': 8,
        'seed': 1,
        'verbosity': 1,
    }
    model = xgb.train(xgb_params, dtrain, num_boost_round=100,
                      evals=watchlist, verbose_eval=False)
    y_pred = model.predict(dval)
    print(f"eta={eta} → RMSE={rmse(y_val, y_pred):.4f}")

eta=0.3 → RMSE=1.8265
eta=0.1 → RMSE=1.7248


### Answer Q6
ans: **`0.1`** — lower RMSE (1.7248 vs 1.8265)

## Final Answers

### Question 1
ans: **`model_year`** — the decision tree's first split at depth=1

### Question 2
ans: Validation RMSE **`1.837`**

### Question 3
ans: Best `n_estimators` = **`150`** (RMSE = 1.769)

### Question 4
ans: Best `max_depth` = **`10`** (mean RMSE = 1.754)

### Question 5
ans: Most important feature = **`vehicle_weight`** (importance = 0.2033)

### Question 6
ans: Better eta = **`0.1`** (RMSE = 1.7248)